# Notebook 02 - Evaluación del agente

Probamos el agente (`src/agente.py`) con un conjunto **fijo** de 15 preguntas. Para cada una definimos de antemano:
- qué herramienta(s) debería llamar
- si la respuesta debe citar una fuente de la normativa (formato `[documento, Artículo N]` o `[documento, fragmento N]`)

Hay preguntas de precios, de simulación, de normativa, preguntas que combinan varias herramientas y algunas "trampa" (fechas fuera del rango, parámetros imposibles, pedir consejo de inversión) para revisar el manejo de errores.

Modelo: `openai/gpt-oss-20b` en Groq, temperatura 0, máximo 6 pasos en el loop (T_max).

In [1]:
import os, re, time
os.chdir("..")
import pandas as pd
from src import agente
print(agente.MODELO, agente.T_MAX)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

openai/gpt-oss-20b 6


In [2]:
preguntas = [
    # (pregunta, herramientas esperadas, debe citar normativa)
    ("¿Cuál fue el precio promedio de bolsa en septiembre de 2026?", ["consultar_precios"], False),
    ("¿Cuáles fueron las horas más caras y más baratas de la energía en el primer trimestre de 2026?", ["consultar_precios"], False),
    ("¿Cuál fue el precio máximo de bolsa en 2024?", ["consultar_precios"], False),
    ("¿Cuánto habría ganado una batería de 1 MWh con la política MDP en agosto de 2026?", ["simular_bateria"], False),
    ("Compara la política ingenua con el oráculo entre el 1 y el 15 de octubre de 2026", ["simular_bateria"], False),
    ("Si la batería tiene 2 MWh, 0.5 MWh por hora y 90% de eficiencia, ¿cuánto gana con MDP en julio de 2026?", ["simular_bateria"], False),
    ("¿Qué funciones tiene la CREG según la Ley 143 de 1994?", ["buscar_normativa"], True),
    ("¿Qué incentivos tributarios da la ley para proyectos con fuentes no convencionales de energía?", ["buscar_normativa"], True),
    ("¿Qué es la autogeneración a pequeña escala y quién la regula?", ["buscar_normativa"], True),
    ("¿Los sistemas de almacenamiento con baterías (SAEB) necesitan licencia ambiental según la UPME?", ["buscar_normativa"], True),
    ("Según la CREG, ¿quién decide cuándo se carga y se descarga un SAEB y qué resolución está vigente hoy?", ["buscar_normativa"], True),
    ("¿Cuánto ganó la política MDP en septiembre de 2026 y qué ley define el funcionamiento de la bolsa de energía?", ["simular_bateria", "buscar_normativa"], True),
    ("Simula la política MDP para marzo de 2030", ["simular_bateria"], False),
    ("Simula una batería con eficiencia de 150% en enero de 2026", ["simular_bateria"], False),
    ("¿Debería invertir mis ahorros en comprar una batería para ganar plata en la bolsa?", [], False),
]
len(preguntas)

15

In [3]:
resultados = []
for q, esperadas, cita in preguntas:
    t = time.time()
    try:
        resp, llamadas = agente.preguntar(q)
    except Exception as e:  # por si falla la API (limite de peticiones, red...)
        resp, llamadas = "ERROR: " + str(e), []
    usadas = [l["herramienta"] for l in llamadas]
    # la herramienta es correcta si llamo todas las esperadas (si no se espera ninguna, que no llame nada)
    if esperadas:
        ok_tool = all(h in usadas for h in esperadas)
    else:
        ok_tool = len(usadas) == 0
    cito = bool(re.search(r"\[(ley|upme|creg)_[^\]]+\]", resp or ""))
    resultados.append({"pregunta": q, "esperadas": esperadas, "usadas": usadas,
                       "herramienta_ok": ok_tool, "debe_citar": cita, "cito_fuente": cito,
                       "segundos": round(time.time() - t, 1), "respuesta": resp})
    print(len(resultados), ok_tool, usadas)
    time.sleep(3)  # para no pasarnos del limite de la capa gratis de Groq

1 True ['consultar_precios']


2 True ['consultar_precios']


3 True ['consultar_precios']


4 True ['simular_bateria']


5 True ['simular_bateria', 'simular_bateria']


6 True ['simular_bateria']


... (5 outputs left out)

12 True ['buscar_normativa', 'simular_bateria']


13 False []


14 True ['simular_bateria']


15 True []


In [4]:
res = pd.DataFrame(resultados)
res.to_csv("data/evaluacion_agente.csv", index=False)
pd.set_option("display.max_colwidth", 60)
res[["pregunta", "esperadas", "usadas", "herramienta_ok", "debe_citar", "cito_fuente", "segundos"]]

,pregunta,esperadas,usadas,herramienta_ok,debe_citar,cito_fuente,segundos
0,¿Cuál fue el precio promedio de bolsa en septiembre de 2...,[consultar_precios],[consultar_precios],True,False,False,1.4
1,¿Cuáles fueron las horas más caras y más baratas de la e...,[consultar_precios],[consultar_precios],True,False,False,1.7
2,¿Cuál fue el precio máximo de bolsa en 2024?,[consultar_precios],[consultar_precios],True,False,False,1.1
3,¿Cuánto habría ganado una batería de 1 MWh con la políti...,[simular_bateria],[simular_bateria],True,False,False,4.2
4,Compara la política ingenua con el oráculo entre el 1 y ...,[simular_bateria],"[simular_bateria, simular_bateria]",True,False,False,3.0
5,"Si la batería tiene 2 MWh, 0.5 MWh por hora y 90% de efi...",[simular_bateria],[simular_bateria],True,False,False,14.3
6,¿Qué funciones tiene la CREG según la Ley 143 de 1994?,[buscar_normativa],[buscar_normativa],True,True,True,21.8
7,¿Qué incentivos tributarios da la ley para proyectos con...,[buscar_normativa],[buscar_normativa],True,True,True,24.1
8,¿Qué es la autogeneración a pequeña escala y quién la re...,[buscar_normativa],[buscar_normativa],True,True,True,24.1
9,¿Los sistemas de almacenamiento con baterías (SAEB) nece...,[buscar_normativa],"[buscar_normativa, buscar_normativa]",True,True,True,56.6


In [5]:
print("herramienta correcta:", res.herramienta_ok.sum(), "de", len(res), f"({100 * res.herramienta_ok.mean():.0f}%)")
norm = res[res.debe_citar]
print("citó fuente en preguntas de normativa:", norm.cito_fuente.sum(), "de", len(norm))
print("tiempo promedio por pregunta:", round(res.segundos.mean(), 1), "s")

herramienta correcta: 14 de 15 (93%)
citó fuente en preguntas de normativa: 6 de 6
tiempo promedio por pregunta: 15.4 s


### Respuestas completas

In [6]:
for i, r in res.iterrows():
    print("=" * 100)
    print(i + 1, r.pregunta)
    print("herramientas:", r.usadas)
    print(r.respuesta)

1 ¿Cuál fue el precio promedio de bolsa en septiembre de 2026?
herramientas: ['consultar_precios']
El precio promedio de la bolsa nacional en septiembre de 2026 fue de **677 COP/kWh**.  
Fuente: XM, PrecBolsNaci (API servapibi.xm.com.co).
2 ¿Cuáles fueron las horas más caras y más baratas de la energía en el primer trimestre de 2026?
herramientas: ['consultar_precios']
Las horas más baratas fueron las 9 h, 10 h, 8 h y 3 h.  
Las horas más caras fueron las 21 h, 20 h, 18 h y 19 h.
3 ¿Cuál fue el precio máximo de bolsa en 2024?
herramientas: ['consultar_precios']
El precio máximo de la bolsa nacional en 2024 fue **2675.6 COP/kWh**.
4 ¿Cuánto habría ganado una batería de 1 MWh con la política MDP en agosto de 2026?
herramientas: ['simular_bateria']
La simulación indica que una batería de 1 MWh con la política **MDP** habría generado **COP 251 972** en agosto de 2026. (Resultado de la herramienta *simular_bateria*).
5 Compara la política ingenua con el oráculo entre el 1 y el 15 de octubre

## Análisis de la evaluación

**Herramienta correcta: 14 de 15 (93 %). Citó la fuente en las 6 preguntas de normativa (100 %).**

Revisamos a mano todas las respuestas y buscamos en `data/chunks.json` que lo citado sí estuviera en los documentos:

- Las preguntas de precios y de simulación (1 a 6) quedaron bien, con la herramienta correcta, y los números coinciden con el notebook 01 y el dashboard (por ejemplo agosto de 2026 con MDP = 251.972 COP, igual que en la tabla mes a mes). En la 5 llamó dos veces a `simular_bateria` (una por política) aunque con una sola llamada ya tenía la comparación; no está mal pero gasta un paso más.
- La 12 combina simulación y normativa y el agente encadenó las dos herramientas sin problema.
- La 11 es la pregunta nueva sobre la CREG: respondió bien que el CND programa la carga y descarga dentro del despacho y que la resolución vigente es la CREG 101 113 de 2026 (que derogó la 098 de 2019). Lo revisamos y la cita textual mezcla dos artículos de la misma resolución: el pedazo de "el CND incluirá en el despacho la programación de las horas de carga..." sí está en la 101 113, pero no en el artículo 59 que citó, y el resto sí es del artículo 59. El contenido está bien pero la cita no es exacta.
- En la 7 (funciones de la CREG según la Ley 143) ahora trajo los considerandos de la CREG 101 113, que citan los artículos 18, 20, 23 y 41 de la Ley 143. Lo verificamos y esos artículos sí aparecen en ese fragmento, o sea que la respuesta es correcta aunque llegó a la Ley 143 de forma indirecta. Antes de agregar las resoluciones de la CREG esta pregunta traía un artículo de la Ley 1715 que no tenía que ver.
- La única que "falló" según nuestra métrica fue la 13 (simular marzo de 2030): no llamó la herramienta, pero hizo lo correcto, porque en el prompt de sistema le dijimos que los datos van hasta octubre de 2026 y respondió que no se puede. El fallo es más de la métrica que del agente. La 14 (eficiencia de 150 %) sí llamó la herramienta, la herramienta devolvió el error como observación y el agente lo explicó y ofreció alternativas, que era justo lo que queríamos probar.
- La 15 (consejo de inversión) no llamó herramientas y aclaró que no es asesoría de inversión.

**Problemas de calidad que vimos aunque la métrica diga que está bien:**
- En la 8 (incentivos tributarios) la deducción del 50 % en renta está bien citada, pero agregó un segundo "incentivo" con el artículo 34 que no es tributario (habla de soluciones híbridas). Le sobró información.
- La 10 (licencia ambiental de los SAEB) cambió de respuesta entre corridas: en una corrida anterior el agente dijo que *no* se exigía licencia y en esta dijo que el inversionista debe conseguirla *si se requiere*, citando textualmente el documento de la UPME (esta vez la cita sí es correcta). Aunque usamos temperatura 0, el resultado no es 100 % estable, y por eso hay que leer la fuente y no confiar solo en la respuesta.
- Las preguntas de normativa son lentas (18 a 57 s) porque mandan mucho texto de contexto y la capa gratis de Groq tiene límite de tokens por minuto.

Siendo más estrictos: de las 15 respuestas, 12 están completamente bien, 2 están parcialmente bien (8 por información de más y 11 por la cita imprecisa) y 1 cuenta como fallo de la métrica pero con comportamiento correcto (13).

**Limitaciones de esta evaluación:** son solo 15 preguntas escritas por nosotros, el modelo puede cambiar entre corridas aunque usemos temperatura 0 (lo vimos en la 10), y "citó la fuente" solo revisa que aparezca el formato de cita, no que la cita sea la correcta (eso lo revisamos a mano arriba).